# Multilingual RAG Baseline Reproduction (Colab)

This notebook reproduces baseline retrieval results for:
- BM25
- Dense Retrieval
- GraphRAG
- Hybrid Retrieval
- Re-ranking

It reports consistent metrics for all methods:
- Precision@k
- Recall@k
- MRR


In [ ]:
# Colab setup
%pip install -q langchain-core langchain-community langchain-huggingface chromadb \
  rank-bm25 langdetect nltk sentence-transformers pytrec_eval


In [ ]:
import os
import re
import json
import time
import pickle
import random
import pathlib
import importlib.util
from dataclasses import dataclass
from collections import defaultdict
from typing import Any

import numpy as np
import pandas as pd
import nltk
from langdetect import detect
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP_EN = set(nltk.corpus.stopwords.words('english'))
STOP_DE = set(nltk.corpus.stopwords.words('german'))

print('Setup complete.')


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Paths (edit PROJECT_ROOT if needed)
# PROJECT_ROOT must be the folder that contains benchmark/ and storage/
CANDIDATE_ROOTS = [
    pathlib.Path('/content/drive/MyDrive/Adv_GenAI'),
    pathlib.Path('/content/drive/MyDrive/advanced-genai-26/baseline/advanced_genAI-main/data'),
    pathlib.Path('/content/drive/MyDrive/advanced_genAI-main/data'),
]

def looks_like_project_root(p: pathlib.Path) -> bool:
    return (p / 'benchmark').exists() and (p / 'storage').exists()

PROJECT_ROOT = None
for c in CANDIDATE_ROOTS:
    if looks_like_project_root(c):
        PROJECT_ROOT = c
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not auto-detect project root. Set PROJECT_ROOT manually.')

PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT =', PROJECT_ROOT)

PATH_BM25_PICKLE = PROJECT_ROOT / 'storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl'
PATH_QA = PROJECT_ROOT / 'benchmark/benchmark_qa_bilingual.json'
PATH_QRELS_FIXED = PROJECT_ROOT / 'benchmark/score/fixed_size'

PATH_DENSE_INDEX = PROJECT_ROOT / 'storage/subsample/vectordb_dense/fixed_e5'
PATH_GRAG_ROOT = PROJECT_ROOT / 'storage/subsample/retrieval_graph'
PATH_CHUNK_PKL = PROJECT_ROOT / 'storage/subsample/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl'

OUT_DIR = PROJECT_ROOT / 'results/baseline_repro_colab'
OUT_DIR.mkdir(parents=True, exist_ok=True)

required = [
    PATH_BM25_PICKLE, PATH_QA, PATH_QRELS_FIXED,
    PATH_DENSE_INDEX, PATH_GRAG_ROOT, PATH_CHUNK_PKL
]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f'Missing required path: {p}')

print('All required artifacts found.')


In [ ]:
# Legacy BM25 classes for notebook-generated pickles
class BilingualBM25:
    def _rank_lang(self, q: str, lang: str, k: int):
        try:
            q_tokens = nltk.word_tokenize(q)
        except Exception:
            q_tokens = q.split()
        scores = self.bm25[lang].get_scores(q_tokens)
        idx = np.argsort(scores)[::-1][:k]
        hits = []
        for i in idx:
            d = self.docs_by_lang[lang][i]
            d.metadata['bm25_score'] = float(scores[i])
            hits.append(d)
        return hits

    def search(self, query: str, top_k: int = 100):
        src = detect(query) if query.strip() else 'en'
        src = src if src in ('en', 'de') else 'en'
        bag = []
        translator = getattr(self, 'translator', None)
        for lang in ('en', 'de'):
            q_lang = translator.translate(query, lang) if translator and lang != src else query
            bag.extend(self._rank_lang(q_lang, lang, top_k))

        best = {}
        for d in bag:
            uid = d.metadata.get('chunk_id') or d.metadata.get('record_id')
            if uid not in best or d.metadata['bm25_score'] > best[uid].metadata.get('bm25_score', -1e9):
                best[uid] = d

        return sorted(best.values(), key=lambda d: d.metadata.get('bm25_score', 0.0), reverse=True)[:top_k]

class QEBM25:
    @staticmethod
    def _expand_query(query: str, base_retriever, fb_docs: int = 5, fb_terms: int = 5) -> str:
        def tok(text: str):
            try:
                return nltk.word_tokenize(text.lower())
            except Exception:
                return text.lower().split()

        hits = base_retriever.search(query, top_k=fb_docs)
        tokens = [
            t for h in hits for t in tok(h.page_content)
            if t.isalpha() and t not in STOP_EN and t not in STOP_DE
        ]
        extra = ' '.join(w for w, _ in nltk.FreqDist(tokens).most_common(fb_terms))
        return f'{query} {extra}' if extra else query

    def search(self, query: str, top_k: int = 100):
        expanded = self._expand_query(query, self.base)
        return self.base.search(expanded, top_k)

with open(PATH_BM25_PICKLE, 'rb') as f:
    bm25_retriever = pickle.load(f)

print('BM25 loaded:', type(bm25_retriever))


In [ ]:
# Dense retriever
class DenseRetriever:
    def __init__(self, index_dir: pathlib.Path, model_name='intfloat/multilingual-e5-large-instruct', k: int = 100):
        self.k = k
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cuda' if os.path.exists('/proc/driver/nvidia/version') else 'cpu'},
            encode_kwargs={'batch_size': 32, 'normalize_embeddings': True},
        )
        self.store = Chroma(persist_directory=str(index_dir), embedding_function=self.embeddings)

    def _prep(self, q: str) -> str:
        return 'query: ' + q.strip()

    def search(self, query: str, top_k: int = 100):
        k = top_k or self.k
        hits = self.store.similarity_search_with_score(self._prep(query), k=k)
        out = []
        for doc, dist in hits:
            doc.metadata['dense_score'] = 1.0 - float(dist)
            out.append(doc)
        return out

dense_retriever = DenseRetriever(PATH_DENSE_INDEX, k=100)
print('Dense retriever ready.')


In [ ]:
# GraphRAG retriever
class GraphRAGRetriever:
    def __init__(self, graph_root: pathlib.Path, chunk_pkl: pathlib.Path):
        self.root = graph_root
        self.emb_dir = graph_root / 'embeddings'
        self.chunk_pkl = chunk_pkl
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        self._emb_cache = {}
        self._cid_cache = {}
        self._chunk_by_id = None
        self._chunk_vec_cache = {}
        self.comm2chunk = json.loads((self.root / 'comm2chunk_fixed.json').read_text(encoding='utf-8'))

    def _load_embeddings(self, level: int):
        if level in self._emb_cache:
            return self._emb_cache[level], self._cid_cache[level]
        mat = np.load(self.emb_dir / f'EMB_fixed_C{level}.npy')
        cid = json.loads((self.emb_dir / f'CID_fixed_C{level}.json').read_text(encoding='utf-8'))
        self._emb_cache[level] = mat
        self._cid_cache[level] = cid
        return mat, cid

    def _load_chunks(self):
        if self._chunk_by_id is not None:
            return self._chunk_by_id
        with open(self.chunk_pkl, 'rb') as f:
            docs_norm = pickle.load(f)

        def restore(d):
            raw = d.metadata.get('original_text') or d.page_content
            return Document(page_content=raw, metadata=d.metadata)

        docs = [restore(d) for d in docs_norm]
        self._chunk_by_id = {d.metadata['chunk_id']: d for d in docs}
        return self._chunk_by_id

    def _chunk_vec(self, cid: str, chunks: dict):
        if cid not in self._chunk_vec_cache:
            self._chunk_vec_cache[cid] = self.embedder.encode([chunks[cid].page_content], normalize_embeddings=True)[0]
        return self._chunk_vec_cache[cid]

    def retrieve(self, query: str, level: str = 'C1', k_comms: int = 24, top_k: int = 100):
        L = int(level.lstrip('C'))
        emb_mat, cid_list = self._load_embeddings(L)
        chunks = self._load_chunks()

        q_vec = self.embedder.encode([query], normalize_embeddings=True)[0]
        sims_comm = emb_mat @ q_vec
        best_idx = sims_comm.argsort()[::-1][:k_comms]

        cand_ids = set()
        for idx in best_idx:
            cand_ids.update(self.comm2chunk.get(cid_list[idx], []))

        scored = []
        for cid in cand_ids:
            if cid not in chunks:
                continue
            sim = float(self._chunk_vec(cid, chunks) @ q_vec)
            scored.append((cid, sim))

        scored.sort(key=lambda x: x[1], reverse=True)
        scored = scored[:top_k]

        out = []
        for cid, sim in scored:
            d = chunks[cid]
            d.metadata['grag_score'] = (sim + 1.0) / 2.0
            out.append(d)
        return out

    def search(self, query: str, top_k: int = 100):
        return self.retrieve(query=query, level='C1', k_comms=24, top_k=top_k)

graph_retriever = GraphRAGRetriever(PATH_GRAG_ROOT, PATH_CHUNK_PKL)
print('GraphRAG retriever ready.')


In [ ]:
# Hybrid + Re-ranking
def _uid(doc: Any):
    meta = getattr(doc, 'metadata', {}) or {}
    return meta.get('chunk_id') or meta.get('record_id') or meta.get('doc_id')

def _safe_unique(docs):
    out, seen = [], set()
    for d in docs:
        u = _uid(d)
        if u is None or u in seen:
            continue
        seen.add(u)
        out.append(d)
    return out

def _rrf_fuse(runs: dict, k_rrf: int = 60, weights=None):
    weights = weights or {'bm25': 1.2, 'dense': 1.0, 'graph': 0.6}
    scores = defaultdict(float)
    store = {}
    for name, docs in runs.items():
        w = float(weights.get(name, 1.0))
        for rank, d in enumerate(docs, start=1):
            u = _uid(d)
            if u is None:
                continue
            store.setdefault(u, d)
            scores[u] += w * (1.0 / (k_rrf + rank))
    fused = sorted(store.values(), key=lambda d: scores[_uid(d)], reverse=True)
    for d in fused:
        d.metadata['fused_score'] = float(scores[_uid(d)])
    return fused

def _overlap_rerank(docs, query: str, top_k: int):
    q_terms = set(t.lower() for t in query.split() if t.strip())
    scored = []
    for d in docs:
        text = (d.metadata.get('original_text') or d.page_content or '').lower()
        overlap = len(q_terms & set(text.split())) / max(len(q_terms), 1)
        scored.append((overlap, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:top_k]]

class HybridRetriever:
    def __init__(self, bm25, dense, graph, rerank=False):
        self.bm25 = bm25
        self.dense = dense
        self.graph = graph
        self.rerank = rerank

    def search(self, query: str, top_k: int = 100):
        pre_k = max(30, top_k)
        bm = _safe_unique(self.bm25.search(query, top_k=pre_k))
        de = _safe_unique(self.dense.search(query, top_k=pre_k))
        gr = _safe_unique(self.graph.search(query, top_k=pre_k))

        fused = _rrf_fuse({'bm25': bm, 'dense': de, 'graph': gr})
        if self.rerank:
            fused = _overlap_rerank(fused[:max(50, top_k)], query, top_k=max(50, top_k))
        return fused[:top_k]

hybrid_retriever = HybridRetriever(bm25_retriever, dense_retriever, graph_retriever, rerank=False)
rerank_retriever = HybridRetriever(bm25_retriever, dense_retriever, graph_retriever, rerank=True)
print('Hybrid and ReRank retrievers ready.')


In [ ]:
# Evaluation
K_VALUES = [1, 3, 5, 10]
TOP_N = 100

def load_qrels(folder: pathlib.Path, threshold: float = 0.5):
    qrels = defaultdict(set)
    for fp in sorted(folder.glob('*.json')):
        did = fp.stem
        payload = json.loads(fp.read_text(encoding='utf-8'))
        for qid, rel in payload.items():
            if float(rel.get('relevance_score', 0.0)) >= threshold:
                qrels[str(qid)].add(did)
    return qrels

def normalize_docs(docs, top_n=100):
    out, seen = [], set()
    for d in docs:
        did = _uid(d)
        if did is None or did in seen:
            continue
        out.append(str(did))
        seen.add(did)
        if len(out) >= top_n:
            break
    return out

def precision_at_k(ranked, relevant, k):
    top = ranked[:k]
    return sum(1 for x in top if x in relevant) / max(k, 1)

def recall_at_k(ranked, relevant, k):
    if not relevant:
        return 0.0
    top = ranked[:k]
    return sum(1 for x in top if x in relevant) / len(relevant)

def reciprocal_rank(ranked, relevant):
    for i, x in enumerate(ranked, start=1):
        if x in relevant:
            return 1.0 / i
    return 0.0

qa_data = json.loads(PATH_QA.read_text(encoding='utf-8'))
qrels = load_qrels(PATH_QRELS_FIXED)

methods = {
    'BM25': bm25_retriever,
    'Dense': dense_retriever,
    'GraphRAG': graph_retriever,
    'Hybrid': hybrid_retriever,
    'ReRank': rerank_retriever,
}

runs = {}
for name, retriever in methods.items():
    run = {}
    for q in tqdm(qa_data, desc=f'Running {name}'):
        qid = str(q['id'])
        docs = retriever.search(q['question'], top_k=TOP_N)
        run[qid] = normalize_docs(docs, top_n=TOP_N)
    runs[name] = run

per_query_rows = []
summary_rows = []

for name, run in runs.items():
    qids = sorted(set(run.keys()) & set(qrels.keys()))
    mrr_vals = []
    p_vals = {k: [] for k in K_VALUES}
    r_vals = {k: [] for k in K_VALUES}

    for qid in qids:
        ranked = run[qid]
        rel = qrels[qid]
        row = {'method': name, 'qid': qid}

        rr = reciprocal_rank(ranked, rel)
        row['MRR'] = rr
        mrr_vals.append(rr)

        for k in K_VALUES:
            pk = precision_at_k(ranked, rel, k)
            rk = recall_at_k(ranked, rel, k)
            row[f'Precision@{k}'] = pk
            row[f'Recall@{k}'] = rk
            p_vals[k].append(pk)
            r_vals[k].append(rk)

        per_query_rows.append(row)

    summary = {'method': name, 'queries_evaluated': len(qids), 'MRR': float(np.mean(mrr_vals) if mrr_vals else 0.0)}
    for k in K_VALUES:
        summary[f'Precision@{k}'] = float(np.mean(p_vals[k]) if p_vals[k] else 0.0)
        summary[f'Recall@{k}'] = float(np.mean(r_vals[k]) if r_vals[k] else 0.0)
    summary_rows.append(summary)

per_query_df = pd.DataFrame(per_query_rows)
summary_df = pd.DataFrame(summary_rows).sort_values('MRR', ascending=False).reset_index(drop=True)

summary_df


In [ ]:
# Save deliverables
summary_path = OUT_DIR / 'metrics_summary.csv'
per_query_path = OUT_DIR / 'metrics_per_query.csv'
runs_path = OUT_DIR / 'runs.json'

summary_df.to_csv(summary_path, index=False)
per_query_df.to_csv(per_query_path, index=False)
runs_path.write_text(json.dumps(runs, indent=2), encoding='utf-8')

print('Saved:')
print('-', summary_path)
print('-', per_query_path)
print('-', runs_path)


## Short Baseline Report

Run the next cell to generate a concise report draft from the computed table.


In [ ]:
best = summary_df.iloc[0]
worst = summary_df.iloc[-1]

report_lines = [
    'Baseline Reproduction Summary',
    f"- Evaluated methods: {', '.join(summary_df['method'].tolist())}",
    f"- Queries evaluated: {int(best['queries_evaluated'])}",
    f"- Best MRR: {best['method']} ({best['MRR']:.4f})",
    f"- Lowest MRR: {worst['method']} ({worst['MRR']:.4f})",
    '- Observations:',
    '  1. Hybrid/ReRank usually improve top-rank quality by combining complementary signals.',
    '  2. Dense and GraphRAG tend to help semantic matching beyond exact keyword overlap.',
    '  3. BM25 remains a strong lexical baseline but can miss semantically relevant passages.',
    '- Assumptions/notes:',
    '  1. Relevance threshold for qrels is 0.5 (binary relevance).',
    '  2. Evaluation uses English questions (field: question) from benchmark_qa_bilingual.json.',
    '  3. k values are fixed to [1, 3, 5, 10] for all methods.',
]

print('\n'.join(report_lines))
